# RF-DETR — Appearance ReID tracking comparison (Colab)

Compare person tracking with appearance **ReID off vs on** over one clip.

The **same detections** are fed through two tracking pipelines that differ only in `reid_enabled`, so any change in track-id stability is attributable to ReID alone. The lightweight ReID is a CPU-friendly HSV torso-color histogram (no embedding model, no extra dependencies).

**How to read the results:** if `mean active` (average people per frame) stays about the same while `unique track ids` drops, ReID is successfully reviving ids for people who left and returned. If the average count itself drops, ids are being wrongly merged — raise `--reid-similarity`.

> Tip: `Runtime → Change runtime type → GPU` before running (keypoint inference is slow on CPU).

In [ ]:
!nvidia-smi -L || echo 'No GPU detected — inference will be slow. Runtime > Change runtime type > GPU.'

## 1. Clone and install

In [ ]:
import os

REPO_URL = 'https://github.com/shingo257/rf-detr.git'
BRANCH = 'develop'

if not os.path.exists('rf-detr'):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd rf-detr
!pip -q install -e .
print('\nInstalled. If imports fail below, use Runtime > Restart session, then re-run from this cell (the clone is skipped automatically).')

## 2. Choose a video

Defaults to the bundled `sample/mzoo.mov`. To use your own clip, uncomment the upload lines.

In [ ]:
VIDEO = 'sample/mzoo.mov'  # bundled sample

# --- To upload your own video instead, uncomment: ---
# from google.colab import files
# uploaded = files.upload()
# VIDEO = next(iter(uploaded))

print('Using video:', VIDEO)

## 3. Run the comparison (ReID off vs on)

The first run downloads the keypoint model weights. `--max-frames` keeps it quick; raise or drop it for your clip.

In [ ]:
!rfdetr-demo compare-reid --source {VIDEO} --max-frames 600 --json reid_metrics.json

import json
print('\n' + json.dumps(json.load(open('reid_metrics.json')), indent=2, ensure_ascii=False))

## 4. Sweep the revival threshold

Find a good `--reid-similarity`: you want `unique ids` to drop **while** `mean active` stays roughly constant. Too low a threshold merges different people (mean drops); too high revives nobody (no change vs off).

In [ ]:
import json, subprocess
import pandas as pd

rows = []
for similarity in [0.3, 0.4, 0.5, 0.6, 0.7]:
    subprocess.run(
        ['rfdetr-demo', 'compare-reid', '--source', VIDEO, '--max-frames', '600',
         '--reid-similarity', str(similarity), '--json', 'sweep.json'],
        check=True,
    )
    data = json.load(open('sweep.json'))
    off, on = data['reid_off'], data['reid_on']
    rows.append({
        'similarity': similarity,
        'ids_off': off['unique_ids'],
        'ids_on': on['unique_ids'],
        'mean_off': round(off['mean_active'], 2),
        'mean_on': round(on['mean_active'], 2),
        'std_off': round(off['count_std'], 2),
        'std_on': round(on['count_std'], 2),
    })

pd.DataFrame(rows)

## 5. Visual check — annotated video with ReID on

Render the annotated MP4 with your chosen setting and play it inline. Watch whether a person who is briefly occluded keeps the same skeleton id when they reappear.

In [ ]:
!RFDETR_TRACK_REID=1 RFDETR_REID_SIMILARITY=0.5 rfdetr-demo video --task keypoint --source {VIDEO} --max-frames 600 --output reid_on.mp4

from base64 import b64encode
from IPython.display import HTML

mp4 = b64encode(open('reid_on.mp4', 'rb').read()).decode()
HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>')

## Tuning cheatsheet

All default off; enable per run via env vars or the `compare-reid` flags.

| Env var | Flag | Meaning | Default |
| --- | --- | --- | --- |
| `RFDETR_TRACK_REID` | (compare-reid always runs both) | Enable appearance ReID | off |
| `RFDETR_REID_WEIGHT` | `--reid-weight` | Appearance vs IoU cost blend (0..1) | 0.3 |
| `RFDETR_REID_SIMILARITY` | `--reid-similarity` | Min histogram match to revive an id | 0.5 |
| `RFDETR_REID_GALLERY_FRAMES` | `--reid-gallery-frames` | How long a retired id stays revivable | 60 |
| `RFDETR_REID_EMA` | — | Descriptor smoothing (0..1) | 0.9 |

If the color histogram proves too weak for your footage (e.g. everyone wears similar colors), the descriptor in `src/rfdetr_demo/tracking/appearance.py` can later be swapped for a small ONNX/OSNet embedding behind the same interface.